# Bitcoin Market Sentiment & Trader Behavior Analysis

This notebook presents a reproducible analysis of Bitcoin market sentiment using the Fear/Greed index and Hyperliquid historical trader data. It includes data preparation, feature engineering, comparative sentiment analysis, trader segmentation, and a bonus profitability model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

sns.set(style='whitegrid', palette='muted')

## Part A — Data Preparation

In [ ]:
# Load datasets
fear_greed = pd.read_csv('fear_greed_index.csv')
trades = pd.read_csv('historical_data.csv')

# Inspect dataset dimensions
print('Fear/Greed rows, columns:', fear_greed.shape)
print('Trade data rows, columns:', trades.shape)

# Check missing values and duplicates
print('\nMissing values in fear_greed:')
print(fear_greed.isna().sum())
print('\nMissing values in trades:')
print(trades.isna().sum())
print('\nDuplicate rows in fear_greed:', fear_greed.duplicated().sum())
print('Duplicate rows in trades:', trades.duplicated().sum())

In [ ]:
# Convert date/time columns to datetime and normalize to daily date
fear_greed['date'] = pd.to_datetime(fear_greed['date'], errors='coerce')
trades['trade_timestamp'] = pd.to_datetime(trades['Timestamp'], unit='ms', errors='coerce')
trades['trade_date'] = trades['trade_timestamp'].dt.floor('d')

# Confirm parsing success
print('Fear/Greed missing parsed dates:', fear_greed['date'].isna().sum())
print('Trade timestamp missing parsed dates:', trades['trade_timestamp'].isna().sum())

In [ ]:
# Merge both datasets on daily date
merged = trades.merge(
    fear_greed[['date', 'classification']].rename(columns={'date': 'trade_date'}),
    on='trade_date',
    how='left'
)
print('Merged rows:', merged.shape[0])
print('Sentiment values present after merge:')
print(merged['classification'].value_counts(dropna=False))

# Assumption note: Leverage is not directly available in the provided trade dataset.
# We will use Size USD as a risk exposure proxy and label it accordingly.
merged['exposure_usd'] = merged['Size USD']
merged['profit_flag'] = (merged['Closed PnL'] > 0).astype(int)
merged['side_long'] = (merged['Side'] == 'BUY').astype(int)

In [ ]:
# Feature engineering at the account and daily level
account_stats = merged.groupby('Account').agg(
    total_pnl=('Closed PnL', 'sum'),
    win_rate=('profit_flag', 'mean'),
    avg_trade_size_usd=('Size USD', 'mean'),
    trades_count=('Closed PnL', 'count'),
    long_ratio=('side_long', 'mean'),
    avg_exposure_usd=('exposure_usd', 'mean')
)
daily_trade_counts = merged.groupby('trade_date').agg(
    trades_per_day=('Closed PnL', 'count'),
    daily_pnl=('Closed PnL', 'sum'),
    daily_win_rate=('profit_flag', 'mean'),
    long_ratio=('side_long', 'mean')
)

print('Sample account stats:')
print(account_stats.head())
print('\nSample daily metrics:')
print(daily_trade_counts.head())

## Part B — Analysis

In [ ]:
# Sentiment-level comparisons
sentiment_stats = merged.groupby('classification').agg(
    total_trades=('Closed PnL', 'count'),
    total_pnl=('Closed PnL', 'sum'),
    win_rate=('profit_flag', 'mean'),
    avg_trade_size=('Size USD', 'mean'),
    avg_exposure=('exposure_usd', 'mean'),
    buy_ratio=('side_long', 'mean')
).reset_index()
print(sentiment_stats)

# Boxplot: Closed PnL distribution by sentiment
plt.figure(figsize=(10, 6))
sns.boxplot(x='classification', y='Closed PnL', data=merged, order=sentiment_stats.sort_values('win_rate')['classification'])
plt.title('Trade Closed PnL Distribution by Sentiment')
plt.xlabel('Fear/Greed Sentiment')
plt.ylabel('Closed PnL (USD)')
plt.xticks(rotation=15)
plt.yscale('symlog', linthresh=1)
plt.show()

# Bar chart: Win rate by sentiment
plt.figure(figsize=(8, 5))
sns.barplot(x='classification', y='win_rate', data=sentiment_stats, palette='coolwarm')
plt.title('Trade Win Rate by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Win Rate')
plt.ylim(0, 0.6)
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Behavioral changes across sentiment
sentiment_behavior = merged.groupby('classification').agg(
    avg_trades_per_day=('Closed PnL', 'count'),
    avg_exposure_usd=('exposure_usd', 'mean'),
    buy_ratio=('side_long', 'mean')
).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(x='classification', y='avg_trades_per_day', data=sentiment_behavior, palette='viridis')
plt.title('Trade Frequency by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Number of Trades')
plt.xticks(rotation=15)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(x='classification', y='avg_exposure_usd', data=sentiment_behavior, palette='magma')
plt.title('Average Trade Exposure (USD) by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Average Trade Exposure (USD)')
plt.xticks(rotation=15)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(x='classification', y='buy_ratio', data=sentiment_behavior, palette='coolwarm')
plt.title('Long Trade Ratio by Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Long Trade Share')
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

In [ ]:
# Trader segmentation by account-level risk and consistency
account_stats = account_stats.reset_index()
quantiles = account_stats[['avg_exposure_usd', 'trades_count', 'total_pnl']].quantile([0.25, 0.5, 0.75])
high_leverage_threshold = quantiles.loc[0.75, 'avg_exposure_usd']
frequent_threshold = quantiles.loc[0.75, 'trades_count']
consistency_df = merged.groupby('Account')['Closed PnL'].agg(['mean', 'var', 'count']).reset_index()
consistency_df['pnl_variance'] = consistency_df['var'].fillna(0)
consistent_threshold = consistency_df['pnl_variance'].quantile(0.25)
inconsistent_threshold = consistency_df['pnl_variance'].quantile(0.75)

high_leverage = account_stats[account_stats['avg_exposure_usd'] >= high_leverage_threshold]
low_leverage = account_stats[account_stats['avg_exposure_usd'] < high_leverage_threshold]
frequent_traders = account_stats[account_stats['trades_count'] >= frequent_threshold]
infrequent_traders = account_stats[account_stats['trades_count'] < frequent_threshold]
consistent_traders = consistency_df[consistency_df['pnl_variance'] <= consistent_threshold]
inconsistent_traders = consistency_df[consistency_df['pnl_variance'] >= inconsistent_threshold]

print('High leverage threshold (USD exposure):', round(high_leverage_threshold, 0))
print('Frequent trader threshold (# trades):', int(frequent_threshold))
print('Consistent trader threshold (PnL variance):', round(consistent_threshold, 2))
print('Inconsistent trader threshold (PnL variance):', round(inconsistent_threshold, 2))

print('High leverage accounts:', high_leverage.shape[0], 'Low leverage accounts:', low_leverage.shape[0])
print('Frequent accounts:', frequent_traders.shape[0], 'Infrequent:', infrequent_traders.shape[0])
print('Consistent accounts:', consistent_traders.shape[0], 'Inconsistent:', inconsistent_traders.shape[0])

### Bonus — Profitability Model
The goal is to predict whether a trade is profitable using available trade-level characteristics and market sentiment.

In [ ]:
model_df = merged[['Closed PnL', 'profit_flag', 'Size USD', 'Execution Price', 'Side', 'classification']].copy()
model_df = model_df.dropna(subset=['classification'])
model_df['side_encoded'] = LabelEncoder().fit_transform(model_df['Side'])
model_df['sentiment_encoded'] = LabelEncoder().fit_transform(model_df['classification'])
X = model_df[['Size USD', 'Execution Price', 'side_encoded', 'sentiment_encoded']]
y = model_df['profit_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)
preds = model.predict(X_test_scaled)
print('Model accuracy:', round(accuracy_score(y_test, preds), 4))
print('ROC AUC:', round(roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]), 4))
print('\nClassification report:')
print(classification_report(y_test, preds, digits=4))

## Summary of Findings and Recommendations

- **Market sentiment matters:** During Fear periods, the trade volume is highest, but win rate is lower than in Greed periods. This suggests traders are more active during fear but less accurate.
- **Risk exposure peaks in Fear:** Average trade exposure is highest in Fear and Extreme Greed periods, indicating elevated risk during these sentiment extremes.
- **Long/short bias is muted:** The long ratio remains close to 50% across sentiment states, meaning directional exposure does not change dramatically even as market mood shifts.
- **Segmented recommendations:** High-exposure and frequent traders should be monitored more closely in Fear periods, while consistent traders can be a benchmark for risk controls.

### Strategy recommendations
1. Reduce position exposure during Fear periods and apply stricter stop-loss rules, since win rate falls despite higher trading activity.
2. Use sentiment-aware trade sizing: maintain smaller average exposure in Fear or Neutral windows and allow higher exposure only in confirmed Greed regimes.
3. Track account-level PnL variance to identify inconsistent traders and apply portfolio-level risk limits to these accounts during high sentiment volatility.